In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import time

SEED = 42
CLASS_INDICES = [207, 281, 963]
CLASS_NAMES = ["golden retriever", "tabby cat", "pizza"]
NUM_STEPS = 25
GUIDANCE_SCALE = 4.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_RUNS = 3
SCHEME = Int8WeightOnlyConfig()

def load_pipeline():
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype = torch.float16,
    )

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    return pipe.to(DEVICE)

base_pipe = load_pipeline()

LAYER_TARGETS = {
    "attn" : ["to_q", "to_k", "to_v", "to_out"],
    "mlp" : ["ff.net"],
    "adaLN" : ["norm1.linear"],
}

def quantize_selective(transformer, targets):
    keywords = []
    if "all" in targets : 
        keywords = None
    else:
        for t in targets:
            keywords.extend(LAYER_TARGETS[t])

    count = 0
    for name, module in transformer.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        if keywords is None or any(k in name for k in keywords):
            quantize_(module, SCHEME)
            count+=1

    return count

def benchmark(pipe, class_indices, num_runs = NUM_RUNS):
    torch.cuda.reset_peak_memory_stats()
    latencies = []
    images = None

    for i in range(num_runs):
        torch.manual_seed(SEED)
        torch.cuda.synchronize()
        t0 = time.time()

        output =  pipe(
            class_labels = class_indices,
            num_inference_steps = NUM_STEPS,
            guidance_scale = GUIDANCE_SCALE,
        )

        torch.cuda.synchronize()
        latencies.append(time.time() - t0)
        images = output.images

    avg_latency = sum(latencies) /num_runs
    peak_mem = torch.cuda.max_memory_allocated() / 1e9

    flags = []
    for img in images:
        mean_px = np.array(img).mean()
        flags.append("BLACK" if mean_px<5.0 else "OK")

    return images, avg_latency, peak_mem, flags

configs = {
    "baseline":       [],           # no quantization
    "adaLN_only":     ["adaLN"],
    "attn_only":      ["attn"],
    "mlp_only":       ["mlp"],
    "attn_mlp":       ["attn", "mlp"],
    "adaLN_attn":     ["adaLN", "attn"],
    "adaLN_mlp":      ["adaLN", "mlp"],
    "all":            ["all"],
}

results = {}

for config_name, targets in configs.items():
    print(f"\n{'='*55}")
    print(f"  Config: {config_name}")
    print(f"{'='*55}")

    pipe = deepcopy(base_pipe).to(DEVICE)

    n_quantized = 0
    if targets:
        try:
            n_quantized = quantize_selective(pipe.transformer, targets)
            print(f"  Quantized {n_quantized} layers")
        except Exception as e:
            print(f"  [FAILED] {e}")
            results[config_name] = None
            del pipe
            torch.cuda.empty_cache()
            continue

    try:
        images, latency, peak_mem, flags = benchmark(pipe, CLASS_INDICES)
        results[config_name] = {
            "images": images,
            "latency": latency,
            "peak_mem": peak_mem,
            "flags": flags,
            "n_quantized": n_quantized,
        }
        print(f"  Latency     : {latency:.2f}s")
        print(f"  Peak mem    : {peak_mem:.2f} GB")
    except Exception as e:
        print(f"  [FAILED] Inference error: {e}")
        results[config_name] = None
    finally:
        del pipe
        torch.cuda.empty_cache()  

print(f"\n\n{'Config':<20} {'Layers':>8} {'Latency':>10} {'Peak Mem':>10} {'Images':>20}")
print("-" * 72)
for name, val in results.items():
    if val is None:
        print(f"{name:<20} {'—':>8} {'—':>10} {'—':>10} {'FAILED':>20}")
    else:
        flags_str = " | ".join(val["flags"])
        print(f"{name:<20} {val['n_quantized']:>8} {val['latency']:>9.2f}s "
              f"{val['peak_mem']:>9.2f}GB  {flags_str:>20}")

valid = {k: v for k, v in results.items() if v is not None}
n_configs = len(valid)
n_classes = len(CLASS_NAMES)

fig, axes = plt.subplots(n_configs, n_classes, figsize=(5 * n_classes, 4 * n_configs))

for row, (config_name, val) in enumerate(valid.items()):
    for col, (img, class_name, flag) in enumerate(zip(val["images"], CLASS_NAMES, val["flags"])):
        ax = axes[row][col] if n_configs > 1 else axes[col]
        ax.imshow(img)
        ax.set_title(f"{class_name} [{flag}]", fontsize=8)
        if col == 0:
            ax.set_ylabel(
                f"{config_name}\n{val['latency']:.2f}s | {val['peak_mem']:.2f}GB",
                fontsize=8, rotation=0, labelpad=120, va="center"
            )
        ax.axis("off")

plt.suptitle(
    f"DiT-XL/2 — Selective int8 Quantization (seed={SEED}, {NUM_STEPS} steps)",
    fontsize=13
)
plt.tight_layout()
plt.savefig("dit_selective_quantization.png", dpi=150, bbox_inches="tight")
plt.show()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
import torch
from torchao.quantization import quantize_, Float8WeightOnlyConfig
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import time

SEED = 42
CLASS_INDICES = [207, 281, 963]
CLASS_NAMES = ["golden retriever", "tabby cat", "pizza"]
NUM_STEPS = 25
GUIDANCE_SCALE = 4.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_RUNS = 3
SCHEME = Float8WeightOnlyConfig()

def load_pipeline():
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype = torch.float16,
    )

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    return pipe.to(DEVICE)

base_pipe = load_pipeline()

LAYER_TARGETS = {
    "attn" : ["to_q", "to_k", "to_v", "to_out"],
    "mlp" : ["ff.net"],
    "adaLN" : ["norm1.linear"],
}

def quantize_selective(transformer, targets):
    keywords = []
    if "all" in targets : 
        keywords = None
    else:
        for t in targets:
            keywords.extend(LAYER_TARGETS[t])

    count = 0
    for name, module in transformer.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        if keywords is None or any(k in name for k in keywords):
            quantize_(module, SCHEME)
            count+=1

    return count

def benchmark(pipe, class_indices, num_runs = NUM_RUNS):
    torch.cuda.reset_peak_memory_stats()
    latencies = []
    images = None

    for i in range(num_runs):
        torch.manual_seed(SEED)
        torch.cuda.synchronize()
        t0 = time.time()

        output =  pipe(
            class_labels = class_indices,
            num_inference_steps = NUM_STEPS,
            guidance_scale = GUIDANCE_SCALE,
        )

        torch.cuda.synchronize()
        latencies.append(time.time() - t0)
        images = output.images

    avg_latency = sum(latencies) /num_runs
    peak_mem = torch.cuda.max_memory_allocated() / 1e9

    flags = []
    for img in images:
        mean_px = np.array(img).mean()
        flags.append("BLACK" if mean_px<5.0 else "OK")

    return images, avg_latency, peak_mem, flags

configs = {
    "baseline":       [],           # no quantization
    "adaLN_only":     ["adaLN"],
    "attn_only":      ["attn"],
    "mlp_only":       ["mlp"],
    "attn_mlp":       ["attn", "mlp"],
    "adaLN_attn":     ["adaLN", "attn"],
    "adaLN_mlp":      ["adaLN", "mlp"],
    "all":            ["all"],
}

results = {}

for config_name, targets in configs.items():
    print(f"\n{'='*55}")
    print(f"  Config: {config_name}")
    print(f"{'='*55}")

    pipe = deepcopy(base_pipe).to(DEVICE)

    n_quantized = 0
    if targets:
        try:
            n_quantized = quantize_selective(pipe.transformer, targets)
            print(f"  Quantized {n_quantized} layers")
        except Exception as e:
            print(f"  [FAILED] {e}")
            results[config_name] = None
            del pipe
            torch.cuda.empty_cache()
            continue

    try:
        images, latency, peak_mem, flags = benchmark(pipe, CLASS_INDICES)
        results[config_name] = {
            "images": images,
            "latency": latency,
            "peak_mem": peak_mem,
            "flags": flags,
            "n_quantized": n_quantized,
        }
        print(f"  Latency     : {latency:.2f}s")
        print(f"  Peak mem    : {peak_mem:.2f} GB")
    except Exception as e:
        print(f"  [FAILED] Inference error: {e}")
        results[config_name] = None
    finally:
        del pipe
        torch.cuda.empty_cache()  

print(f"\n\n{'Config':<20} {'Layers':>8} {'Latency':>10} {'Peak Mem':>10} {'Images':>20}")
print("-" * 72)
for name, val in results.items():
    if val is None:
        print(f"{name:<20} {'—':>8} {'—':>10} {'—':>10} {'FAILED':>20}")
    else:
        flags_str = " | ".join(val["flags"])
        print(f"{name:<20} {val['n_quantized']:>8} {val['latency']:>9.2f}s "
              f"{val['peak_mem']:>9.2f}GB  {flags_str:>20}")

valid = {k: v for k, v in results.items() if v is not None}
n_configs = len(valid)
n_classes = len(CLASS_NAMES)

fig, axes = plt.subplots(n_configs, n_classes, figsize=(5 * n_classes, 4 * n_configs))

for row, (config_name, val) in enumerate(valid.items()):
    for col, (img, class_name, flag) in enumerate(zip(val["images"], CLASS_NAMES, val["flags"])):
        ax = axes[row][col] if n_configs > 1 else axes[col]
        ax.imshow(img)
        ax.set_title(f"{class_name} [{flag}]", fontsize=8)
        if col == 0:
            ax.set_ylabel(
                f"{config_name}\n{val['latency']:.2f}s | {val['peak_mem']:.2f}GB",
                fontsize=8, rotation=0, labelpad=120, va="center"
            )
        ax.axis("off")

plt.suptitle(
    f"DiT-XL/2 — Selective int8 Quantization (seed={SEED}, {NUM_STEPS} steps)",
    fontsize=13
)
plt.tight_layout()
plt.savefig("dit_selective_quantization.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import torch
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import time

SEED = 42
CLASS_INDICES = [207, 281, 963]
CLASS_NAMES = ["golden retriever", "tabby cat", "pizza"]
NUM_STEPS = 25
GUIDANCE_SCALE = 4.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_RUNS = 3
SCHEME =  Int8DynamicActivationInt8WeightConfig()

def load_pipeline():
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype = torch.float16,
    )

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    return pipe.to(DEVICE)

base_pipe = load_pipeline()

LAYER_TARGETS = {
    "attn" : ["to_q", "to_k", "to_v", "to_out"],
    "mlp" : ["ff.net"],
    "adaLN" : ["norm1.linear"],
}

def quantize_selective(transformer, targets):
    keywords = []
    if "all" in targets : 
        keywords = None
    else:
        for t in targets:
            keywords.extend(LAYER_TARGETS[t])

    count = 0
    for name, module in transformer.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        if keywords is None or any(k in name for k in keywords):
            quantize_(module, SCHEME)
            count+=1

    return count

def benchmark(pipe, class_indices, num_runs = NUM_RUNS):
    torch.cuda.reset_peak_memory_stats()
    latencies = []
    images = None

    for i in range(num_runs):
        torch.manual_seed(SEED)
        torch.cuda.synchronize()
        t0 = time.time()

        output =  pipe(
            class_labels = class_indices,
            num_inference_steps = NUM_STEPS,
            guidance_scale = GUIDANCE_SCALE,
        )

        torch.cuda.synchronize()
        latencies.append(time.time() - t0)
        images = output.images

    avg_latency = sum(latencies) /num_runs
    peak_mem = torch.cuda.max_memory_allocated() / 1e9

    flags = []
    for img in images:
        mean_px = np.array(img).mean()
        flags.append("BLACK" if mean_px<5.0 else "OK")

    return images, avg_latency, peak_mem, flags

configs = {
    "baseline":       [],           # no quantization
    "adaLN_only":     ["adaLN"],
    "attn_only":      ["attn"],
    "mlp_only":       ["mlp"],
    "attn_mlp":       ["attn", "mlp"],
    "adaLN_attn":     ["adaLN", "attn"],
    "adaLN_mlp":      ["adaLN", "mlp"],
    "all":            ["all"],
}

results = {}

for config_name, targets in configs.items():
    print(f"\n{'='*55}")
    print(f"  Config: {config_name}")
    print(f"{'='*55}")

    pipe = deepcopy(base_pipe).to(DEVICE)

    n_quantized = 0
    if targets:
        try:
            n_quantized = quantize_selective(pipe.transformer, targets)
            print(f"  Quantized {n_quantized} layers")
        except Exception as e:
            print(f"  [FAILED] {e}")
            results[config_name] = None
            del pipe
            torch.cuda.empty_cache()
            continue

    try:
        images, latency, peak_mem, flags = benchmark(pipe, CLASS_INDICES)
        results[config_name] = {
            "images": images,
            "latency": latency,
            "peak_mem": peak_mem,
            "flags": flags,
            "n_quantized": n_quantized,
        }
        print(f"  Latency     : {latency:.2f}s")
        print(f"  Peak mem    : {peak_mem:.2f} GB")
    except Exception as e:
        print(f"  [FAILED] Inference error: {e}")
        results[config_name] = None
    finally:
        del pipe
        torch.cuda.empty_cache()  

print(f"\n\n{'Config':<20} {'Layers':>8} {'Latency':>10} {'Peak Mem':>10} {'Images':>20}")
print("-" * 72)
for name, val in results.items():
    if val is None:
        print(f"{name:<20} {'—':>8} {'—':>10} {'—':>10} {'FAILED':>20}")
    else:
        flags_str = " | ".join(val["flags"])
        print(f"{name:<20} {val['n_quantized']:>8} {val['latency']:>9.2f}s "
              f"{val['peak_mem']:>9.2f}GB  {flags_str:>20}")

valid = {k: v for k, v in results.items() if v is not None}
n_configs = len(valid)
n_classes = len(CLASS_NAMES)

fig, axes = plt.subplots(n_configs, n_classes, figsize=(5 * n_classes, 4 * n_configs))

for row, (config_name, val) in enumerate(valid.items()):
    for col, (img, class_name, flag) in enumerate(zip(val["images"], CLASS_NAMES, val["flags"])):
        ax = axes[row][col] if n_configs > 1 else axes[col]
        ax.imshow(img)
        ax.set_title(f"{class_name} [{flag}]", fontsize=8)
        if col == 0:
            ax.set_ylabel(
                f"{config_name}\n{val['latency']:.2f}s | {val['peak_mem']:.2f}GB",
                fontsize=8, rotation=0, labelpad=120, va="center"
            )
        ax.axis("off")

plt.suptitle(
    f"DiT-XL/2 — Selective int8 Quantization (seed={SEED}, {NUM_STEPS} steps)",
    fontsize=13
)
plt.tight_layout()
plt.savefig("dit_selective_quantization.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import torch
from torchao.quantization import quantize_, Int8DynamicActivationInt4WeightConfig
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import time

SEED = 42
CLASS_INDICES = [207, 281, 963]
CLASS_NAMES = ["golden retriever", "tabby cat", "pizza"]
NUM_STEPS = 25
GUIDANCE_SCALE = 4.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_RUNS = 3
SCHEME =  Int8DynamicActivationInt4WeightConfig()

def load_pipeline():
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype = torch.float16,
    )

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    return pipe.to(DEVICE)

base_pipe = load_pipeline()

LAYER_TARGETS = {
    "attn" : ["to_q", "to_k", "to_v", "to_out"],
    "mlp" : ["ff.net"],
    "adaLN" : ["norm1.linear"],
}

def quantize_selective(transformer, targets):
    keywords = []
    if "all" in targets : 
        keywords = None
    else:
        for t in targets:
            keywords.extend(LAYER_TARGETS[t])

    count = 0
    for name, module in transformer.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        if keywords is None or any(k in name for k in keywords):
            quantize_(module, SCHEME)
            count+=1

    return count

def benchmark(pipe, class_indices, num_runs = NUM_RUNS):
    torch.cuda.reset_peak_memory_stats()
    latencies = []
    images = None

    for i in range(num_runs):
        torch.manual_seed(SEED)
        torch.cuda.synchronize()
        t0 = time.time()

        output =  pipe(
            class_labels = class_indices,
            num_inference_steps = NUM_STEPS,
            guidance_scale = GUIDANCE_SCALE,
        )

        torch.cuda.synchronize()
        latencies.append(time.time() - t0)
        images = output.images

    avg_latency = sum(latencies) /num_runs
    peak_mem = torch.cuda.max_memory_allocated() / 1e9

    flags = []
    for img in images:
        mean_px = np.array(img).mean()
        flags.append("BLACK" if mean_px<5.0 else "OK")

    return images, avg_latency, peak_mem, flags

configs = {
    "baseline":       [],           # no quantization
    "adaLN_only":     ["adaLN"],
    "attn_only":      ["attn"],
    "mlp_only":       ["mlp"],
    "attn_mlp":       ["attn", "mlp"],
    "adaLN_attn":     ["adaLN", "attn"],
    "adaLN_mlp":      ["adaLN", "mlp"],
    "all":            ["all"],
}

results = {}

for config_name, targets in configs.items():
    print(f"\n{'='*55}")
    print(f"  Config: {config_name}")
    print(f"{'='*55}")

    pipe = deepcopy(base_pipe).to(DEVICE)

    n_quantized = 0
    if targets:
        try:
            n_quantized = quantize_selective(pipe.transformer, targets)
            print(f"  Quantized {n_quantized} layers")
        except Exception as e:
            print(f"  [FAILED] {e}")
            results[config_name] = None
            del pipe
            torch.cuda.empty_cache()
            continue

    try:
        images, latency, peak_mem, flags = benchmark(pipe, CLASS_INDICES)
        results[config_name] = {
            "images": images,
            "latency": latency,
            "peak_mem": peak_mem,
            "flags": flags,
            "n_quantized": n_quantized,
        }
        print(f"  Latency     : {latency:.2f}s")
        print(f"  Peak mem    : {peak_mem:.2f} GB")
    except Exception as e:
        print(f"  [FAILED] Inference error: {e}")
        results[config_name] = None
    finally:
        del pipe
        torch.cuda.empty_cache()  

print(f"\n\n{'Config':<20} {'Layers':>8} {'Latency':>10} {'Peak Mem':>10} {'Images':>20}")
print("-" * 72)
for name, val in results.items():
    if val is None:
        print(f"{name:<20} {'—':>8} {'—':>10} {'—':>10} {'FAILED':>20}")
    else:
        flags_str = " | ".join(val["flags"])
        print(f"{name:<20} {val['n_quantized']:>8} {val['latency']:>9.2f}s "
              f"{val['peak_mem']:>9.2f}GB  {flags_str:>20}")

valid = {k: v for k, v in results.items() if v is not None}
n_configs = len(valid)
n_classes = len(CLASS_NAMES)

fig, axes = plt.subplots(n_configs, n_classes, figsize=(5 * n_classes, 4 * n_configs))

for row, (config_name, val) in enumerate(valid.items()):
    for col, (img, class_name, flag) in enumerate(zip(val["images"], CLASS_NAMES, val["flags"])):
        ax = axes[row][col] if n_configs > 1 else axes[col]
        ax.imshow(img)
        ax.set_title(f"{class_name} [{flag}]", fontsize=8)
        if col == 0:
            ax.set_ylabel(
                f"{config_name}\n{val['latency']:.2f}s | {val['peak_mem']:.2f}GB",
                fontsize=8, rotation=0, labelpad=120, va="center"
            )
        ax.axis("off")

plt.suptitle(
    f"DiT-XL/2 — Selective int8 Quantization (seed={SEED}, {NUM_STEPS} steps)",
    fontsize=13
)
plt.tight_layout()
plt.savefig("dit_selective_quantization.png", dpi=150, bbox_inches="tight")
plt.show()